In [0]:
customer_df = spark.table("workspace.default.bronze_customers")

display(customer_df)

customer_id,city,subscription_type,age,signup_date,operation,update_ts,source_file,ingestion_timestamp
4,Delhi,Basic,28,2024-01-29,INSERT,2026-04-06T08:38:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
18,Chennai,Basic,27,2024-05-12,DELETE,2026-04-02T06:56:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
29,Chennai,Premium,47,2024-09-12,DELETE,2026-04-02T06:32:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
28,Mumbai,Premium,49,2024-12-21,INSERT,2026-04-06T15:15:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
18,Delhi,Premium,34,2024-01-02,INSERT,2026-04-01T21:15:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
70,Bangalore,Basic,25,2024-12-01,UPDATE,2026-04-05T11:39:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
12,Hyderabad,Premium,50,2024-05-16,INSERT,2026-04-01T19:08:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
5,Chennai,Basic,44,2024-05-25,UPDATE,2026-04-02T18:45:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
95,Chennai,Basic,22,2024-04-01,INSERT,2026-04-01T09:17:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
70,Bangalore,Basic,37,2024-03-23,INSERT,2026-04-02T05:53:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z


In [0]:
customer_df = customer_df.dropDuplicates(["customer_id"])

In [0]:
(
    customer_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.silver_customers")
)

In [0]:
bronze_customers = spark.table("workspace.default.bronze_customers")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("customer_id").orderBy(
    col("update_ts").desc()
)

latest_customers = (
    bronze_customers
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
display(latest_customers)

customer_id,city,subscription_type,age,signup_date,operation,update_ts,source_file,ingestion_timestamp
4,Hyderabad,Premium,37,2024-10-31,INSERT,2026-04-07T22:32:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
5,Delhi,Basic,24,2024-02-16,INSERT,2026-04-07T12:11:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
12,Hyderabad,Basic,23,2024-01-27,INSERT,2026-04-07T21:23:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
14,Chennai,Basic,24,2024-02-27,UPDATE,2026-04-07T19:56:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
15,Delhi,Basic,35,2024-10-23,INSERT,2026-04-07T02:00:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
18,Delhi,Basic,47,2024-02-09,INSERT,2026-04-07T22:30:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
26,Bangalore,Premium,42,2024-09-11,UPDATE,2026-04-06T18:13:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
28,Mumbai,Premium,40,2024-07-08,DELETE,2026-04-07T10:01:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
29,Hyderabad,Basic,34,2024-08-07,UPDATE,2026-04-06T23:03:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
30,Chennai,Basic,28,2024-02-17,DELETE,2026-04-07T08:33:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z


In [0]:
(
    latest_customers.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.silver_customers")
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM workspace.default.silver_customers
    ORDER BY customer_id
    LIMIT 20
    """)
)

customer_id,city,subscription_type,age,signup_date,operation,update_ts,source_file,ingestion_timestamp
4,Hyderabad,Premium,37,2024-10-31,INSERT,2026-04-07T22:32:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
5,Delhi,Basic,24,2024-02-16,INSERT,2026-04-07T12:11:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
12,Hyderabad,Basic,23,2024-01-27,INSERT,2026-04-07T21:23:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
14,Chennai,Basic,24,2024-02-27,UPDATE,2026-04-07T19:56:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
15,Delhi,Basic,35,2024-10-23,INSERT,2026-04-07T02:00:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
18,Delhi,Basic,47,2024-02-09,INSERT,2026-04-07T22:30:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
26,Bangalore,Premium,42,2024-09-11,UPDATE,2026-04-06T18:13:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
28,Mumbai,Premium,40,2024-07-08,DELETE,2026-04-07T10:01:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
29,Hyderabad,Basic,34,2024-08-07,UPDATE,2026-04-06T23:03:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
30,Chennai,Basic,28,2024-02-17,DELETE,2026-04-07T08:33:00.000Z,dbfs:/Volumes/workspace/default/data/customer_cdc_data_final.csv,2026-07-11T19:12:11.817Z
